# When Does a Word Become a Concept?
## Diachronic embeddings of *culture* in nineteenth-century British newspapers

This notebook contains the calculations and visualisations used in the article  
**«Когда слово становится понятием? Диахронические векторные модели в исследовании культуры (на материале британской прессы XIX века)»**.

The analysis uses the published decade-level word embeddings from the *Living with Machines* project (1800s–1910s):  
Zenodo record **7181682**.

Methodological reference:  
Pedrazzini N., McGillivray B. *Machines in the Media: Semantic Change in the Lexicon of Mechanization in 19th-Century British Newspapers*. NLP4DH, 2022. DOI: 10.18653/v1/2022.nlp4dh-1.12.

## Setup

In [ ]:
!pip -q install "gensim>=4.3,<5" "ruptures>=1.1,<2"

In [ ]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from tqdm.auto import tqdm

import scipy.linalg
if not hasattr(scipy.linalg, "triu"):
    scipy.linalg.triu = np.triu

from gensim.models import KeyedVectors
import ruptures as rpt
from IPython.display import display

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

## Data and parameters

In [ ]:
PERIODS = [
    "1800s", "1810s", "1820s", "1830s", "1840s", "1850s",
    "1860s", "1870s", "1880s", "1890s", "1900s", "1910s"
]
REFERENCE = "1910s"
EARLIER = PERIODS[:-1]

ZENODO_RECORD = 7181682
MODEL_DIR = Path("/content/lwm_vectors")
MODEL_DIR.mkdir(exist_ok=True)

AGRICULTURAL_FIELD = [
    "soil", "agriculture", "agricultural",
    "land", "farming", "crop", "husbandry"
]

INTELLECTUAL_FIELD = [
    "education", "learning", "knowledge",
    "literature", "art", "science", "taste"
]

CONTROL_WORDS = [
    "mother", "father", "horse", "bread", "water", "church",
    "winter", "morning", "village", "garden", "friend", "dinner"
]

CULTURE_DECADES = ["1800s", "1840s", "1850s", "1860s", "1870s"]
CIVILISATION_DECADES = ["1820s", "1840s", "1860s", "1880s", "1910s"]

TOPN = 30

In [ ]:
def download_model(period):
    path = MODEL_DIR / f"{period}-vectors.txt"
    if path.exists():
        return path

    url = (
        f"https://zenodo.org/records/{ZENODO_RECORD}/files/"
        f"{period}-vectors.txt?download=1"
    )

    with requests.get(url, stream=True, timeout=(30, 600)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with path.open("wb") as f, tqdm(
            total=total if total else None,
            unit="B",
            unit_scale=True,
            desc=period
        ) as bar:
            for chunk in response.iter_content(chunk_size=4 << 20):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

    return path

model_paths = {period: download_model(period) for period in PERIODS}

## Analysis

In [ ]:
def load_model(period):
    model = KeyedVectors.load_word2vec_format(
        str(model_paths[period]),
        binary=False,
        unicode_errors="ignore"
    )
    model.fill_norms(force=True)
    return model

def vector(model, word):
    if word not in model:
        return None
    return model.get_vector(word, norm=True)

def cosine(v1, v2):
    if v1 is None or v2 is None:
        return np.nan
    return float(np.dot(v1, v2))

def mean_similarity(model, target, words):
    target_vector = vector(model, target)
    similarities = [
        cosine(target_vector, vector(model, word))
        for word in words
        if word in model
    ]
    return float(np.mean(similarities))

def neighbors(model, word, topn=TOPN):
    return model.most_similar(word, topn=topn)

def neighbor_table(df, decades):
    selected = df[df["decade"].isin(decades)].copy()
    selected["item"] = (
        selected["neighbor"]
        + " ("
        + selected["similarity"].map(lambda x: f"{x:.3f}")
        + ")"
    )
    return (
        selected.pivot(index="rank", columns="decade", values="item")
        .reindex(columns=decades)
    )

In [ ]:
reference_model = load_model(REFERENCE)
reference_words = ["culture"] + CONTROL_WORDS
reference_vectors = {
    word: vector(reference_model, word)
    for word in reference_words
}
del reference_model
gc.collect()

shift_rows = []
field_rows = []
relation_rows = []
culture_neighbor_rows = []
civilisation_neighbor_rows = []

for period in tqdm(PERIODS, desc="Decades"):
    model = load_model(period)

    if period in EARLIER:
        for word in reference_words:
            shift_rows.append({
                "decade": period,
                "word": word,
                "cosine_to_1910s": cosine(
                    vector(model, word),
                    reference_vectors[word]
                )
            })

    field_rows.append({
        "decade": period,
        "agricultural_cultivation": mean_similarity(
            model, "culture", AGRICULTURAL_FIELD
        ),
        "education_intellectual_life": mean_similarity(
            model, "culture", INTELLECTUAL_FIELD
        )
    })

    culture_vector = vector(model, "culture")
    relation_rows.append({
        "decade": period,
        "culture_cultivation": cosine(
            culture_vector, vector(model, "cultivation")
        ),
        "culture_civilisation": cosine(
            culture_vector, vector(model, "civilisation")
        )
    })

    for rank, (word, similarity) in enumerate(
        neighbors(model, "culture"), start=1
    ):
        culture_neighbor_rows.append({
            "decade": period,
            "rank": rank,
            "neighbor": word,
            "similarity": similarity
        })

    for rank, (word, similarity) in enumerate(
        neighbors(model, "civilisation"), start=1
    ):
        civilisation_neighbor_rows.append({
            "decade": period,
            "rank": rank,
            "neighbor": word,
            "similarity": similarity
        })

    del model
    gc.collect()

shift_df = pd.DataFrame(shift_rows)
fields_df = pd.DataFrame(field_rows).set_index("decade")
relations_df = pd.DataFrame(relation_rows).set_index("decade")
culture_neighbors_df = pd.DataFrame(culture_neighbor_rows)
civilisation_neighbors_df = pd.DataFrame(civilisation_neighbor_rows)

## Temporal trajectory

In [ ]:
shift_pivot = shift_df.pivot(
    index="decade",
    columns="word",
    values="cosine_to_1910s"
).reindex(EARLIER)

culture_shift = shift_pivot["culture"]
control_mean = shift_pivot[CONTROL_WORDS].mean(axis=1)

display(pd.DataFrame({
    "culture → 1910s": culture_shift,
    "control mean": control_mean
}).round(3))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(EARLIER, culture_shift, marker="o", linewidth=2.2, label="culture → 1910s")
ax.plot(EARLIER, control_mean, marker="s", linewidth=2.0, label="Control mean")
ax.set_xlabel("Decade")
ax.set_ylabel("Cosine similarity")
ax.set_title("Culture relative to the 1910s model")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
values = culture_shift.to_numpy()

for penalty in [0.25, 0.50]:
    changepoints = rpt.Pelt(
        model="l1",
        jump=1,
        min_size=2
    ).fit(values).predict(pen=penalty)

    internal = [point for point in changepoints if point < len(values)]

    if internal:
        transitions = [
            f"{EARLIER[point - 1]} → {EARLIER[point]}"
            for point in internal
        ]
        print(f"PELT, penalty={penalty}: {', '.join(transitions)}")
    else:
        print(f"PELT, penalty={penalty}: no internal changepoint")

## Nearest neighbours of *culture*

In [ ]:
display(neighbor_table(culture_neighbors_df, CULTURE_DECADES))

## Diagnostic semantic fields

In [ ]:
field_table = fields_df.rename(columns={
    "agricultural_cultivation": "Agriculture / cultivation",
    "education_intellectual_life": "Education / intellectual life"
})

display(field_table.round(3))

fig, ax = plt.subplots(figsize=(9.5, 5.3))
ax.plot(
    PERIODS,
    fields_df["agricultural_cultivation"],
    marker="o",
    linewidth=2.4,
    label="Agriculture / cultivation"
)
ax.plot(
    PERIODS,
    fields_df["education_intellectual_life"],
    marker="s",
    linewidth=2.4,
    label="Education / intellectual life"
)
ax.set_xlabel("Decade")
ax.set_ylabel("Mean cosine similarity")
ax.set_title("Contextual relations of culture")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## *Culture*, *cultivation*, and *civilisation*

In [ ]:
relation_table = relations_df.rename(columns={
    "culture_cultivation": "culture ↔ cultivation",
    "culture_civilisation": "culture ↔ civilisation"
})

display(relation_table.round(3))

fig, ax = plt.subplots(figsize=(9.5, 5.3))
ax.plot(
    PERIODS,
    relations_df["culture_cultivation"],
    marker="o",
    linewidth=2.4,
    label="culture ↔ cultivation"
)
ax.plot(
    PERIODS,
    relations_df["culture_civilisation"],
    marker="s",
    linewidth=2.4,
    label="culture ↔ civilisation"
)
ax.set_xlabel("Decade")
ax.set_ylabel("Cosine similarity")
ax.set_title("Key relations of culture")
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## Nearest neighbours of *civilisation*

In [ ]:
display(neighbor_table(civilisation_neighbors_df, CIVILISATION_DECADES))